# ODL Assignment 2 — Task 1
## Choosing the best training algorithm (optimizer) at default settings — tested on BOTH models

**The plan, in the order we agreed:**
1. **Architecture first** — we build the two models the assignment needs: an **MLP**
   and a **1D-CNN**. (Justified from the data's nature in Task 2.)
2. **Optimizer race** — train each model with **six** optimizers at their **default**
   settings and see which trains best. This is the "try every cooking method first"
   step.
3. Take the winning optimizer forward as the **baseline** (no tuning yet).
4. Hyperparameter-tune it.
5. Train the final models and report accuracy / precision / recall.

This notebook covers steps 1–2 and picks the optimizer.

**Why we do it properly (not the lecturer's habit of "just use Adam"):**
- Every optimizer trains the **same** model, at **default** settings → a fair race.
- Each optimizer runs **5 times with different random seeds** → we measure the
  *average* and the *consistency*, not one lucky run.
- We record accuracy **and loss** at **every epoch** → we can see learning happen.
- Crucially, we run the race on **both** the MLP **and** the CNN. A single lucky
  architecture can mislead you — and as you will see, it does.

Each step is its own cell with comments, so you can run and read it one at a time.

---
## Step 1 — Import libraries

In [ ]:
# Data tools
import os, json, time
import numpy as np
import pandas as pd

# Deep-learning framework
import tensorflow as tf

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

print("TensorFlow version:", tf.__version__)

---
## Step 2 — Load the ALREADY-PREPARED data

We do **not** re-clean the raw `train.csv`. Task 2 already produced the cleaned,
encoded, scaled files `train_processed.csv` and `val_processed.csv`. We load those
directly so the optimizer race uses the exact same inputs the final models will use.

In [ ]:
# Find the folder holding the processed CSVs, wherever the notebook is opened from
DATA_DIR = None
for base in [".", "Task2_DataPrep", "../Task2_DataPrep", "Airline Satisfaction/Task2_DataPrep"]:
    if os.path.exists(os.path.join(base, "train_processed.csv")):
        DATA_DIR = base
        break
assert DATA_DIR is not None, "Could not find train_processed.csv next to this notebook."
print("Using data folder:", DATA_DIR)

train = pd.read_csv(os.path.join(DATA_DIR, "train_processed.csv"))
val   = pd.read_csv(os.path.join(DATA_DIR, "val_processed.csv"))
print("Training rows:", len(train), "| Validation rows:", len(val))

---
## Step 3 — Separate inputs (X) from the answer (y)

`X` = the 22 feature columns. `y` = the target `satisfaction` (0/1).
The CNN needs its input shaped as a short sequence, so we also make a reshaped copy
`X_*_cnn` of shape (rows, 22, 1) — think of the 22 features laid out in a line so the
convolution can slide across them.

In [ ]:
# MLP inputs: a flat row of 22 numbers
X_train = train.drop(columns="satisfaction").values.astype("float32")
y_train = train["satisfaction"].values
X_val   = val.drop(columns="satisfaction").values.astype("float32")
y_val   = val["satisfaction"].values
NFEAT = X_train.shape[1]

# CNN inputs: same numbers, reshaped to (rows, 22, 1) = a length-22 sequence
X_train_cnn = X_train.reshape((-1, NFEAT, 1))
X_val_cnn   = X_val.reshape((-1, NFEAT, 1))

print("MLP input shape:", X_train.shape, "| CNN input shape:", X_train_cnn.shape)

---
## Step 4 — Build the MLP (model 1)

A small multi-layer perceptron: two hidden layers that read the 22 features and one
output neuron giving the probability the passenger is satisfied. We pass the optimizer
in as an argument so we can swap it without changing anything else.

In [ ]:
def build_mlp(optimizer):
    """Same MLP every time; only the optimizer differs."""
    m = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(NFEAT,)),           # 22 features
        tf.keras.layers.Dense(64, activation="relu"),     # hidden layer 1
        tf.keras.layers.Dense(32, activation="relu"),     # hidden layer 2
        tf.keras.layers.Dense(1,  activation="sigmoid"),  # P(satisfied)
    ])
    m.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
    return m

---
## Step 5 — Build the 1D-CNN (model 2)

A convolutional network reads the 22 features as a short sequence and uses small
sliding filters to pick up *local* patterns between neighbouring features. Two Conv1D
layers find patterns, `GlobalMaxPooling1D` keeps the strongest signal from each filter,
then a Dense layer makes the final decision.

In [ ]:
def build_cnn(optimizer):
    """Same 1D-CNN every time; only the optimizer differs."""
    m = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(NFEAT, 1)),                       # 22-long sequence
        tf.keras.layers.Conv1D(32, 3, activation="relu", padding="same"),
        tf.keras.layers.Conv1D(64, 3, activation="relu", padding="same"),
        tf.keras.layers.GlobalMaxPooling1D(),                          # strongest signal per filter
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(1,  activation="sigmoid"),               # P(satisfied)
    ])
    m.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
    return m

---
## Step 6 — Define the SIX optimizers at their DEFAULT settings

The six contenders, each at its library-default learning rate (no hand-tuning — that
would make the race unfair). A fresh optimizer object is needed for every run, so we
wrap them in a function.

| Optimizer | Plain-English idea |
|---|---|
| **SGD** | Basic. Small steps straight downhill. Slow. |
| **SGD+Momentum** | SGD that builds speed like a ball rolling downhill. |
| **RMSprop** | Adapts the step size per weight. |
| **Adam** | RMSprop + momentum. The popular default. |
| **Nadam** | Adam with smarter look-ahead momentum. |
| **AdamW** | Adam with cleaner weight-decay (regularisation). |

In [ ]:
def make_optimizer(name):
    """Return a fresh optimizer at its default learning rate."""
    o = tf.keras.optimizers
    return {
        "SGD":          o.SGD(learning_rate=0.01),
        "SGD+Momentum": o.SGD(learning_rate=0.01, momentum=0.9),
        "RMSprop":      o.RMSprop(learning_rate=0.001),
        "Adam":         o.Adam(learning_rate=0.001),
        "Nadam":        o.Nadam(learning_rate=0.001),
        "AdamW":        o.AdamW(learning_rate=0.001),
    }[name]

OPTIMIZERS = ["SGD", "SGD+Momentum", "RMSprop", "Adam", "Nadam", "AdamW"]
print("Optimizers to compare:", OPTIMIZERS)

---
## Step 7 — One function that runs the whole multi-seed race for a model

To avoid copy-pasting the loop twice, we write it once and call it for each model.
For every optimizer and every seed it trains for 30 fixed epochs (so all curves line
up) and records, at **every epoch**, both **validation accuracy** and **validation
loss**, plus how long the run took.

**Why record loss too?** Accuracy is blunt — it just checks if the prediction crossed
0.5. Loss measures *how confident and correct* the model is, so it can separate two
optimizers that look tied on accuracy.

In [ ]:
def run_race(build_fn, Xtr, Xva, seeds=(0,1,2,3,4), epochs=30, batch=512):
    """Train `build_fn` with all six optimizers x each seed; record acc+loss per epoch."""
    hist = {n: {"acc": [], "loss": [], "best_acc": [], "min_loss": [], "time": []}
            for n in OPTIMIZERS}
    for name in OPTIMIZERS:
        for seed in seeds:
            tf.keras.utils.set_random_seed(seed)            # reproducible run
            model = build_fn(make_optimizer(name))          # fresh model + optimizer
            t0 = time.time()
            h = model.fit(Xtr, y_train, validation_data=(Xva, y_val),
                          epochs=epochs, batch_size=batch, verbose=0)
            hist[name]["acc"].append(h.history["val_accuracy"])   # curve
            hist[name]["loss"].append(h.history["val_loss"])      # curve
            hist[name]["best_acc"].append(max(h.history["val_accuracy"]))
            hist[name]["min_loss"].append(min(h.history["val_loss"]))
            hist[name]["time"].append(time.time() - t0)
        m = np.mean(hist[name]["best_acc"]) * 100
        s = np.std(hist[name]["best_acc"]) * 100
        print(f"  {name:14s} acc={m:6.3f}%  std={s:.3f}%  loss={np.mean(hist[name]['min_loss']):.4f}")
    return hist

---
## Step 8 — Run the race on the MLP

> ⚠️ This trains 6 optimizers × 5 seeds = 30 runs. A few minutes on CPU. Run once.

In [ ]:
print("MLP race:")
mlp_hist = run_race(build_mlp, X_train, X_val)

---
## Step 9 — Run the race on the CNN

> ⚠️ The CNN is heavier — this is the slow cell (roughly 3× the MLP). Run once and
> let it finish.

In [ ]:
print("CNN race:")
cnn_hist = run_race(build_cnn, X_train_cnn, X_val_cnn)

---
## Step 10 — Results tables for each model

For each model: mean accuracy (how high), std (how consistent), min loss (quality of
the predictions), epochs-to-target (how fast it converged), and seconds per run.

In [ ]:
def epochs_to(mean_curve, bar):
    """First epoch (1-indexed) the mean curve reaches `bar` %, else '>30'."""
    for i, v in enumerate(mean_curve):
        if v * 100 >= bar:
            return i + 1
    return ">30"

def summarize(hist, bar):
    rows = []
    for n in OPTIMIZERS:
        acc  = np.array(hist[n]["best_acc"]) * 100
        loss = np.array(hist[n]["min_loss"])
        mean_curve = np.array(hist[n]["acc"]).mean(0)
        rows.append({
            "Optimizer":       n,
            "Mean acc %":      round(acc.mean(), 3),
            "Std %":           round(acc.std(), 3),
            "Min loss":        round(loss.mean(), 4),
            f"Epochs to {bar}%": epochs_to(mean_curve, bar),
            "s/run":           round(np.mean(hist[n]["time"]), 1),
        })
    return pd.DataFrame(rows).sort_values("Mean acc %", ascending=False).reset_index(drop=True)

print("MLP results:")
mlp_table = summarize(mlp_hist, 95); display(mlp_table)
print("\nCNN results:")
cnn_table = summarize(cnn_hist, 92); display(cnn_table)

---
## Step 11 — Graph 1: learning curves — accuracy AND loss, on BOTH models

Four panels. Top row = accuracy climbing (higher is better); bottom row = loss falling
(lower is better); left column = MLP, right column = CNN. Each line is the mean of 5
runs; the shaded band is ±1 std (thin = consistent, fat = unreliable).

**What to look for:** SGD crawling far behind everywhere; the four adaptive optimizers
bunched at the top; and — on the CNN — RMSprop going jagged and unstable, which is a
real mark against it.

In [ ]:
palette = {"SGD": "#C44E52", "SGD+Momentum": "#DD8452", "RMSprop": "#8172B3",
           "Adam": "#4C72B0", "Nadam": "#55A868", "AdamW": "#937860"}

def plot_curves(ax, hist, key, scale, ylabel, ylim, title):
    # exact value per optimizer (best accuracy, or lowest loss) to print in the legend
    val = {n: (np.mean(hist[n]["best_acc"])*100 if key=="acc" else np.mean(hist[n]["min_loss"]))
           for n in OPTIMIZERS}
    fmt = "{:.2f}%" if key=="acc" else "{:.3f}"
    for n in OPTIMIZERS:
        arr = np.array(hist[n][key]) * scale         # (seeds, epochs)
        mean, std = arr.mean(0), arr.std(0)
        ep = np.arange(1, len(mean) + 1)
        ax.plot(ep, mean, label=f"{n}  ({fmt.format(val[n])})", color=palette[n], linewidth=1.9)
        ax.fill_between(ep, mean - std, mean + std, color=palette[n], alpha=0.12)
    ax.set_title(title, fontweight="bold"); ax.set_xlabel("Training epoch")
    ax.set_ylabel(ylabel); ax.set_ylim(*ylim)
    # order the legend best-first so the exact values read like a ranking
    h, l = ax.get_legend_handles_labels()
    order = sorted(range(len(OPTIMIZERS)), key=lambda i: val[OPTIMIZERS[i]], reverse=(key=="acc"))
    ax.legend([h[i] for i in order], [l[i] for i in order], fontsize=8,
              title=("best accuracy" if key=="acc" else "lowest loss"))

fig, ax = plt.subplots(2, 2, figsize=(15, 10))
plot_curves(ax[0,0], mlp_hist, "acc", 100, "Val accuracy (%)", (85, 96.5), "MLP — accuracy (higher better)")
plot_curves(ax[0,1], cnn_hist, "acc", 100, "Val accuracy (%)", (83, 94),   "CNN — accuracy (higher better)")
plot_curves(ax[1,0], mlp_hist, "loss", 1, "Val loss", (0.08, 0.30), "MLP — loss (lower better)")
plot_curves(ax[1,1], cnn_hist, "loss", 1, "Val loss", (0.15, 0.45), "CNN — loss (lower better)")
fig.suptitle("Optimizer race on both models — mean of 5 runs, shaded = ±1 std", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0,0,1,0.97])
plt.savefig("opt_curves_both.png", dpi=140, bbox_inches="tight"); plt.show()

---
## Step 12 — Graph 2: final accuracy on each model, side by side

This is the graph that reveals the twist: the optimizer that tops the **MLP** is not
the one that tops the **CNN**. Blue bars = MLP, green bars = CNN, error bar = ±1 std.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6.5))
x = np.arange(len(OPTIMIZERS)); w = 0.38
mlp_m = [np.mean(mlp_hist[n]["best_acc"])*100 for n in OPTIMIZERS]
mlp_s = [np.std(mlp_hist[n]["best_acc"])*100 for n in OPTIMIZERS]
cnn_m = [np.mean(cnn_hist[n]["best_acc"])*100 for n in OPTIMIZERS]
cnn_s = [np.std(cnn_hist[n]["best_acc"])*100 for n in OPTIMIZERS]
ax.bar(x - w/2, mlp_m, w, yerr=mlp_s, capsize=4, label="MLP", color="#4C72B0")
ax.bar(x + w/2, cnn_m, w, yerr=cnn_s, capsize=4, label="CNN", color="#55A868")
for i in range(len(OPTIMIZERS)):
    ax.text(x[i]-w/2, mlp_m[i]+mlp_s[i]+0.15, f"{mlp_m[i]:.2f}", ha="center", fontsize=8)
    ax.text(x[i]+w/2, cnn_m[i]+cnn_s[i]+0.15, f"{cnn_m[i]:.2f}", ha="center", fontsize=8)
ax.set_title("Final accuracy per model — the winner differs by architecture", fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(OPTIMIZERS); ax.set_ylabel("Best val accuracy (%)")
ax.set_ylim(83, 97); ax.legend()
plt.tight_layout(); plt.savefig("opt_cross_architecture.png", dpi=140, bbox_inches="tight"); plt.show()

---
## Step 12b — Graph 3: the computational cost (time)

Accuracy and loss show how *well* each optimizer learns, but not how much *computing*
it costs. That matters, because the tuning stage trains a model many times over, so a
slow method wastes time on every run. The left panel below shows seconds for one full
30-epoch run. The right panel shows something more useful, the time each optimizer
needed to actually reach a sensible target (95% for the MLP, 92% for the CNN), which
rewards fast learners and exposes slow ones like SGD.

In [ ]:
# time to reach a target accuracy = (seconds per run / 30 epochs) * epochs_to_target
def time_to_target(hist, bar, n):
    e = epochs_to(np.array(hist[n]["acc"]).mean(0), bar)   # reuse epochs_to from Step 10
    return None if e == ">30" else e * np.mean(hist[n]["time"]) / 30

mlp_run = [np.mean(mlp_hist[n]["time"]) for n in OPTIMIZERS]
cnn_run = [np.mean(cnn_hist[n]["time"]) for n in OPTIMIZERS]
mlp_tt  = [time_to_target(mlp_hist, 95, n) for n in OPTIMIZERS]
cnn_tt  = [time_to_target(cnn_hist, 92, n) for n in OPTIMIZERS]

fig, (axA, axB) = plt.subplots(1, 2, figsize=(15, 6))
x = np.arange(len(OPTIMIZERS)); w = 0.38

# Panel A: raw seconds per 30-epoch run
axA.bar(x - w/2, mlp_run, w, label="MLP", color="#4C72B0")
axA.bar(x + w/2, cnn_run, w, label="CNN", color="#55A868")
axA.set_title("Time for one full 30-epoch run", fontweight="bold")
axA.set_xticks(x); axA.set_xticklabels(OPTIMIZERS, rotation=20, ha="right")
axA.set_ylabel("Seconds per run"); axA.legend()

# Panel B: time to reach the target accuracy (None = never reached -> label it)
def draw(ax, vals, off, color, label):
    used = False
    for i, v in enumerate(vals):
        if v is None:
            ax.text(x[i]+off, 0.5, "did not\nreach", ha="center", va="bottom",
                    fontsize=7, color="#C44E52", fontweight="bold")
        else:
            ax.bar(x[i]+off, v, w, color=color, label=None if used else label); used = True
draw(axB, mlp_tt, -w/2, "#4C72B0", "MLP (to 95%)")
draw(axB, cnn_tt, +w/2, "#55A868", "CNN (to 92%)")
axB.set_title("Time to actually reach the target accuracy", fontweight="bold")
axB.set_xticks(x); axB.set_xticklabels(OPTIMIZERS, rotation=20, ha="right")
axB.set_ylabel("Seconds to reach target"); axB.legend()

fig.suptitle("Computational cost of each optimizer on both models", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0,0,1,0.95])
plt.savefig("opt_time_cost.png", dpi=140, bbox_inches="tight"); plt.show()

# Read the timing: architecture drives cost, and among the adaptive trio time is a near-tie
print("Seconds per run  (MLP | CNN):")
for i, n in enumerate(OPTIMIZERS):
    print(f"  {n:14s} {mlp_run[i]:5.1f} | {cnn_run[i]:5.1f}")

---
## Step 13 — Break the tie with cross-architecture robustness

The adaptive optimizers are tied within noise on each model, and the *peak* winner
flips between models. So the right question is not "who won once?" but **"who is
reliably near the top on BOTH models?"** We rank each optimizer on each model and
average the ranks — the lowest average rank is the most robust all-round choice.

In [ ]:
def ranks(table):
    d = table.copy(); d["rk"] = d["Mean acc %"].rank(ascending=False)
    return dict(zip(d["Optimizer"], d["rk"]))
rm, rc = ranks(mlp_table), ranks(cnn_table)

robust = pd.DataFrame([{
    "Optimizer": n, "MLP rank": int(rm[n]), "CNN rank": int(rc[n]),
    "Avg rank": (rm[n]+rc[n])/2,
} for n in OPTIMIZERS]).sort_values("Avg rank").reset_index(drop=True)
display(robust)

print("Reading it: the adaptive trio (Adam, Nadam, AdamW) share the best average rank.")
print("Adam WINS the CNN and ties the MLP; AdamW is 2nd on BOTH (never slips);")
print("Nadam WINS the MLP but drops to 3rd on the CNN.")

---
## Step 14 — The decision

**What the evidence says, honestly:**

- **SGD is out.** Slowest to learn, worst accuracy and loss on both models.
- **SGD+Momentum is better but still behind** the adaptive group on both.
- **RMSprop is fine on the MLP but weak on the CNN** (jagged, oscillating training
  path, and it finished ~0.8% below the three leaders, down near SGD+Momentum).
- **Adam, Nadam, and AdamW are a statistical tie.** The gaps between them are smaller
  than the run-to-run wobble, so nobody truly "won" on accuracy.

**Three measures agree, and none of them separates the adaptive trio:** accuracy
could not split Adam, Nadam, and AdamW, loss only shuffled them slightly, and Step 12b
showed their training time is a near-tie too. Time draws its hard line against SGD, not
inside the leading group.

**The winner differs by architecture, so we pick per model, not one-size-fits-all:**
- **MLP → Nadam.** Highest mean accuracy (95.69%) with the tightest run-to-run spread,
  its loss level with the group and its time identical.
- **CNN → Adam.** Highest accuracy (92.96%) *and* the lowest loss on that model, with
  time matching its rivals. Nadam, which led the MLP, slips to third here.
- **AdamW** shadows both leaders without beating either, so it is the natural fallback
  if later tuning shows overfitting, thanks to its built-in weight decay.

This is why running BOTH models mattered: on the MLP alone we would have crowned Nadam
for everything, and the CNN proved that wrong. If one optimizer ever had to serve both,
Adam is that choice. We carry **Nadam for the MLP** and **Adam for the CNN** into Task 2.

In [ ]:
# State the per-model choice straight from the evidence
mlp_winner = mlp_table.iloc[0]["Optimizer"]   # top mean accuracy on the MLP
cnn_winner = cnn_table.iloc[0]["Optimizer"]   # top mean accuracy on the CNN
print(f"MLP baseline optimizer -> {mlp_winner}  "
      f"({mlp_table.iloc[0]['Mean acc %']}% mean, std {mlp_table.iloc[0]['Std %']}%)")
print(f"CNN baseline optimizer -> {cnn_winner}  "
      f"({cnn_table.iloc[0]['Mean acc %']}% mean, std {cnn_table.iloc[0]['Std %']}%)")
print("\nAdaptive trio are tied on accuracy, loss and time; each model takes its own winner.")
print("Single most-robust optimizer if one were required for both: Adam.")

---
## Conclusion — what goes in the report

> Six optimizers were compared at default settings on both required models (MLP and
> 1D-CNN). To avoid conclusions from a single lucky run, each was trained five times
> with different seeds, recording accuracy and loss at every epoch. SGD was clearly
> weakest on both models, and SGD with momentum trailed the adaptive methods. RMSprop
> matched the leaders on the MLP but trained unsteadily and finished below them on the
> CNN. Adam, Nadam, and AdamW
> were statistically tied on accuracy, on loss, and on training time, but the peak
> performer differed by architecture, so each model was given the optimizer that led it.
> **Nadam** was chosen for the multilayer perceptron and **Adam** for the convolutional
> network, with AdamW as an equally-defensible fallback should tuning reveal overfitting.

Use `opt_curves_both.png`, `opt_cross_architecture.png`, and `opt_time_cost.png` as the
evidence figures.

**Next (below):** build the two basic models with their chosen optimizers, check them
for overfitting, and evaluate them on the sealed test set with the full metric set.

---
# Part 2 — Building and evaluating the two basic models

## Step 15 — Train the two baselines with their chosen optimizers

Now we build the actual basic models: the MLP with **Nadam** and the CNN with **Adam**,
every other setting left at default. We train on the 83,123 training rows, watch the
20,781 validation rows each epoch, and keep a single fixed seed so the run repeats. The
sealed 25,976-row test set is untouched until Step 16.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, roc_curve)

# load the sealed test set (train/val already loaded in Step 2)
test = pd.read_csv(os.path.join(DATA_DIR, "test_processed.csv"))
X_test = test.drop(columns="satisfaction").values.astype("float32")
y_test = test["satisfaction"].values
X_test_cnn = X_test.reshape(-1, NFEAT, 1)

SEED = 42
baseline = {}   # will hold history + trained model for each architecture
for name, build_fn, Xtr, Xva in [("MLP", build_mlp, X_train, X_val),
                                  ("CNN", build_cnn, X_train_cnn, X_val_cnn)]:
    tf.keras.utils.set_random_seed(SEED)
    opt = make_optimizer("Nadam" if name == "MLP" else "Adam")   # chosen per model
    model = build_fn(opt)
    h = model.fit(Xtr, y_train, validation_data=(Xva, y_val),
                  epochs=40, batch_size=512, verbose=0)
    baseline[name] = {"model": model, "history": h.history}
    print(f"{name} trained: final val accuracy = {h.history['val_accuracy'][-1]*100:.2f}%")

---
## Step 16 — Evaluate on the sealed test set (five metrics)

We score each model on data it has never seen. Alongside accuracy, precision, recall
and F1 we add three measures that see angles the others miss: **specificity** (the
true-negative rate), **ROC-AUC** (threshold-free ranking quality), and **MCC** (the
Matthews correlation coefficient, which uses all four confusion-matrix cells and only
scores high when both classes are handled well).

In [ ]:
from sklearn.metrics import matthews_corrcoef

def evaluate(name, Xtest):
    model = baseline[name]["model"]
    proba = model.predict(Xtest, verbose=0).ravel()
    pred = (proba >= 0.5).astype(int)
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        "Accuracy":    accuracy_score(y_test, pred)*100,
        "Precision":   precision_score(y_test, pred)*100,
        "Recall":      recall_score(y_test, pred)*100,
        "Specificity": tn/(tn+fp)*100,                 # true-negative rate (mirror of recall)
        "F1-score":    f1_score(y_test, pred)*100,
        "MCC":         matthews_corrcoef(y_test, pred),  # uses all 4 cells; -1..+1
        "ROC-AUC":     roc_auc_score(y_test, proba),
    }, cm, roc_curve(y_test, proba), proba

metrics, cms, rocs = {}, {}, {}
for name, Xtest in [("MLP", X_test), ("CNN", X_test_cnn)]:
    m, cm, roc, _ = evaluate(name, Xtest)
    metrics[name], cms[name], rocs[name] = m, cm, roc

results_table = pd.DataFrame(metrics).round(3)
results_table.loc["ROC-AUC"] = results_table.loc["ROC-AUC"].round(3)
print("Test-set performance:")
display(results_table)

---
## Step 17 — Overfitting check (train vs validation curves)

A model overfits when it memorises the training data: training accuracy keeps rising
while validation accuracy stalls or falls, opening a gap. We plot both, and compute the
train-to-test accuracy gap. A small gap means the model generalises.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(14, 9))
color = {"MLP": "#4C72B0", "CNN": "#55A868"}
for j, name in enumerate(["MLP", "CNN"]):
    h = baseline[name]["history"]; ep = np.arange(1, len(h["accuracy"])+1)
    ax[0,j].plot(ep, np.array(h["accuracy"])*100, color=color[name], label="Training")
    ax[0,j].plot(ep, np.array(h["val_accuracy"])*100, color=color[name], ls="--", label="Validation")
    ax[0,j].set_title(f"{name} — accuracy (train vs val)", fontweight="bold")
    ax[0,j].set_xlabel("Epoch"); ax[0,j].set_ylabel("Accuracy (%)"); ax[0,j].legend()
    ax[1,j].plot(ep, h["loss"], color=color[name], label="Training")
    ax[1,j].plot(ep, h["val_loss"], color=color[name], ls="--", label="Validation")
    ax[1,j].set_title(f"{name} — loss (train vs val)", fontweight="bold")
    ax[1,j].set_xlabel("Epoch"); ax[1,j].set_ylabel("Loss"); ax[1,j].legend()
plt.tight_layout(); plt.savefig("eval_training_curves.png", dpi=140, bbox_inches="tight"); plt.show()

# train-to-test accuracy gap: small gap = generalises well, no overfitting
for name, Xtr, Xtest in [("MLP", X_train, X_test), ("CNN", X_train_cnn, X_test_cnn)]:
    tr = accuracy_score(y_train, (baseline[name]["model"].predict(Xtr, verbose=0).ravel()>=0.5))
    te = metrics[name]["Accuracy"]/100
    print(f"{name}: train {tr*100:.2f}%  test {te*100:.2f}%  gap {(tr-te)*100:+.2f}%")

---
## Step 18 — Confusion matrices and ROC curves

In [ ]:
import seaborn as sns
# confusion matrices
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5)); labels = ["Not satisfied", "Satisfied"]
for i, name in enumerate(["MLP", "CNN"]):
    sns.heatmap(cms[name], annot=True, fmt=",d", cbar=False, ax=ax[i],
                cmap="Blues" if name=="MLP" else "Greens",
                xticklabels=labels, yticklabels=labels)
    ax[i].set_title(f"{name} — confusion matrix (test)", fontweight="bold")
    ax[i].set_xlabel("Predicted"); ax[i].set_ylabel("Actual")
plt.tight_layout(); plt.savefig("eval_confusion.png", dpi=140, bbox_inches="tight"); plt.show()

# ROC curves
fig, axr = plt.subplots(figsize=(8, 7))
for name, c in [("MLP", "#4C72B0"), ("CNN", "#55A868")]:
    fpr, tpr, _ = rocs[name]
    axr.plot(fpr, tpr, color=c, lw=2.3, label=f"{name} (AUC = {metrics[name]['ROC-AUC']:.3f})")
axr.plot([0,1],[0,1], color="#B0B0B0", ls="--", label="Random guess")
axr.set_xlabel("False positive rate"); axr.set_ylabel("True positive rate")
axr.set_title("ROC curves on the sealed test set", fontweight="bold"); axr.legend(loc="lower right")
plt.tight_layout(); plt.savefig("eval_roc.png", dpi=140, bbox_inches="tight"); plt.show()

---
## Step 19 — Conclusion of the basic model evaluation

Both baselines are sound and neither overfits (train-to-test gaps under 0.7%). The
**MLP wins on every metric** — accuracy, precision, recall, specificity, F1, MCC and ROC-AUC — because a
fully connected network reads all 22 features together, while the CNN's sliding filters
assume a local structure that flat tabular data does not strongly hold. The MLP is
therefore the stronger basic model and is carried into Task 2 for hyperparameter
tuning; the CNN stays as a comparison point.